# 05 · Evaluación y Conclusiones

**Fase CRISP-DM:** 5–6 de 6 — Evaluation & Lessons Learned  
¿El proyecto responde lo que se propuso? ¿Qué NO responde? Protocolo de verificación cruzada y limitaciones.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

import hashlib
import pandas as pd
from constantes import ARCHIVO_COTAS_RAW

df = pd.read_csv(ARCHIVO_COTAS_RAW, parse_dates=["fecha"])

## 1. Verificación de reproducibilidad

Cualquier persona debe poder validar que el dataset publicado no fue alterado: la huella del archivo y el recuento de filas se comparan con lo producido por el scraper.

In [2]:
sha256 = hashlib.sha256(ARCHIVO_COTAS_RAW.read_bytes()).hexdigest()
print(f"archivo: {ARCHIVO_COTAS_RAW.name}")
print(f"sha256 : {sha256}")
print(f"filas  : {len(df)} | rango: {df['fecha'].min().date()} → {df['fecha'].max().date()}")
print(f"ultima consulta del scraper: {df['fecha_consulta'].max()}")

archivo: cotas_historico.csv
sha256 : ab84228d0ea4d708b552336f5c105e69fb842dfb37dd38f41ae2edb95126773d
filas  : 4899 | rango: 2022-01-01 → 2026-08-15
ultima consulta del scraper: 2026-08-16T05:22:24+00:00


## 2. Protocolo de verificación cruzada con prensa independiente

El proyecto contrasta **datos oficiales contra comunicaciones oficiales**. Como triangulación opcional (no automatizada), se propone este protocole manual:

1. Elegir un episodio visible en la serie (p. ej. salida de banda, caída pronunciada).
2. Buscar cobertura de prensa de ese período en medios ecuatorianos de registro.
3. Registrar en `data/comunicados/comunicados.csv` las declaraciones **oficiales** citadas (con enlace al medio Y, cuando exista, al comunicado original de la institución).
4. Re-ejecutar los notebooks 03 y 04: las fechas quedarán marcadas sobre la serie y la tabla de contexto mostrará la cota medida de cada día.

**Regla de oro:** la tabla de contexto presenta números; no etiqueta a nadie. Si un comunicado y una cota difieren, el lector dispone del enlace para verificar y sacar su propia conclusión.

## 3. Limitaciones documentadas

| # | Limitación | Impacto | Mitigación |
|---|------------|---------|------------|
| L1 | La cota diaria es la de medianoche local; no captura mínimos intradía | Un comunicado vespertino puede referirse a un nivel no exactamente igual al del CSV | El endpoint horario (`pointValues`) queda documentado para consultas puntuales |
| L2 | Certificado TLS autofirmado en el puerto 8443 de la API | No se puede validar la cadena de certificados | Verificación cruzada con el tablero público y respaldo en web.archive.org |
| L3 | Los umbrales son valores de referencia declarados, no una norma publicada de acceso directo en un solo documento oficial | Pequeño riesgo de desactualización | Están versionados en `src/constantes.py` con su procedencia anotada |
| L4 | El registro de comunicados es manual y voluntario | Cobertura parcial de declaraciones | Reglas de registro explícitas y neutras (cualquier signo) en `data/comunicados/LEEME.md` |
| L5 | Serie desde 2022-01-01 (límite verificado de la API) | No hay historia previa | El backfill puede re-intentarse hacia atrás si la API amplía su retención |
| L6 | El archivo en web.archive.org es mejor-esfuerzo | Puede haber días sin snapshot | Cada intento queda registrado en `data/raw/archivo_web.log` |

## 4. ¿Qué responde este proyecto y qué no?

**Responde:**
- ¿Cuál fue la cota medida de Mazar, Amaluza y Sopladora en cualquier fecha desde 2022?
- ¿Cuándo estuvo la cota dentro o fuera de la banda normal de operación?
- ¿Qué cota media, mínima y máxima rodea la fecha de cada comunicado registrado?

**No responde (por diseño):**
- Si una institución mintió o se equivocó: eso es juicio; aquí solo hay superposición de datos.
- Si un nivel es "bueno" o "malo" para el sistema eléctrico: eso requiere análisis operativo fuera del alcance.
- Predicciones hidrológicas: no hay modelo predictivo y no hace falta para el objetivo.

## 5. Estado actual del dataset

In [3]:
estado = df.groupby("embalse").agg(
    primera=("fecha", "min"),
    ultima=("fecha", "max"),
    n=("fecha", "size"),
)
estado

,primera,ultima,n
embalse,,,
Amaluza,2022-01-01,2026-08-15,1633
Mazar,2022-01-01,2026-08-15,1633
Sopladora,2022-01-01,2026-08-15,1633


## 6. Lecciones aprendidas y próximos pasos

- **El dato público existe pero no es obvio:** la API ORDS detrás del tablero Angular no está documentada; este proyecto documenta su contrato para que cualquiera pueda auditarlo.
- **La automatización es barata y civil:** pocas peticiones diarias no significan carga para el servidor y producen un registro público continuo.

**Próximos pasos cumplidos (actualizados 2026-08):**

1. ~~Registrar los primeros comunicados y ejecutar el cruce completo~~ — el cruce está implementado (notebooks 03–04); falta el registro manual de declaraciones.
2. ~~Incorporar caudales~~ — los `mrid` de caudal ya están en `src/constantes.py`; su serie diaria queda como trabajo futuro.
3. ✅ **Contexto de generación (CENACE) integrado:** `src/generacion.py` recolecta diariamente la composición porcentual del tablero de Información Operativa (con salvedad documentada sobre magnitudes absolutas).
4. ✅ **estado.json público:** banderas factuales por embalse consumibles por sistemas de terceros (`docs/estado.json`).
5. ✅ **Factsheets trimestrales:** una página imprimible por trimestre con los hechos medidos (`docs/factsheets/`).
6. ✅ **Sección de episodios:** rachas ≥3 días fuera de banda o bajo nivel crítico, generadas desde el dato.

**Trabajo futuro:** serie de caudales, calibración de las magnitudes del tablero CENACE contra informes oficiales, y registro ciudadano de comunicados.